# Mêmes lettres, contenus différents · *Same letters, different contents*

Notebook compagnon du chapitre **25. Masse monétaire M1, M2 : ce que ces agrégats mesurent vraiment** — [lire l'article](https://nmlab.io/ressources/masse-monetaire-m1-m2).
Companion notebook to chapter **25. Money Supply M1, M2: What These Aggregates Really Measure** — [read the article](https://nmlab.io/en/ressources/money-supply-m1-m2).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure est régénérée par le code — un **schéma éditable** : changez les libellés à votre guise. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure from code — an **editable diagram**: change the labels as you like; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


# (schéma : aucune donnée externe)


import numpy as np
import pandas as pd
from matplotlib.figure import Figure
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

C, W = nm.COLORS, nm.WIDTH_PX

EA_M3_KEY = "BSI/M.U2.Y.V.M30.X.1.U2.2300.Z01.E"   # encours M3 zone euro (BCE)


def load(series_id: str, start: str | None = None, end: str | None = None) -> pd.Series:
    """Charge une série en direct : FRED, ou le portail de la BCE pour « EA_M3 »."""
    if series_id == "EA_M3":
        url = (f"https://data-api.ecb.europa.eu/service/data/{EA_M3_KEY}"
               "?format=csvdata&detail=dataonly")
        raw = pd.read_csv(url)
        s = pd.Series(raw["OBS_VALUE"].values,
                      index=pd.PeriodIndex(raw["TIME_PERIOD"], freq="M").to_timestamp())
        s = s.sort_index() / 1000.0                 # millions -> milliards d'euros
    else:
        s = nm.load_fred(series_id)
    return s.loc[start:end]


def T(d: dict, lang: str):
    """Sélectionne le jeu de libellés de la langue demandée."""
    return d[lang]


def build_figure(lang: str = "fr") -> Figure:
    """Construit la figure NMLab du chapitre (libellés selon ``lang``)."""
    fig = nm.figure(1140); ax = nm.blank_axes(fig)
    d = dict(fr=("Mêmes lettres, contenus différents","Où loge le livret d'épargne ? Dans M1 aux États-Unis, dans M2 en zone euro.",
                 "États-Unis","Zone euro",
                 [("Numéraire",False),("Dépôts à vue",False),("Épargne (livrets)",True),
                  ("Dépôts à terme courts",False),("Fonds monétaires",False)],
                 [("Numéraire",False),("Dépôts à vue",False),("Épargne (préavis ≤ 3 mois)",True),
                  ("Dépôts à terme ≤ 2 ans",False),("Repos, OPCVM, titres courts",False)],
                 ["M1","M1","M1","M2","M2"],["M1","M1","M2","M2","M3"],
                 "L'épargne (surlignée) bascule d'un agrégat à l'autre selon le continent. Sources : FRED, BCE."),
             en=("Same letters, different contents","Where do savings sit? In M1 in the U.S., in M2 in the euro area.",
                 "United States","Euro area",
                 [("Currency",False),("Demand deposits",False),("Savings deposits",True),
                  ("Small time deposits",False),("Money-market funds",False)],
                 [("Currency",False),("Overnight deposits",False),("Savings (notice ≤ 3 months)",True),
                  ("Time deposits ≤ 2 years",False),("Repos, MMF shares, short paper",False)],
                 ["M1","M1","M1","M2","M2"],["M1","M1","M2","M2","M3"],
                 "Savings (highlighted) jump from one aggregate to the other by continent. Sources: FRED, ECB."))
    t=T(d,lang); nm.header(fig,t[0],t[1])
    def col(cx,title,items,tags):
        ax.text(cx,900,title,ha="center",va="center",fontsize=25,fontweight="bold",color=C["text"])
        y=820; h=110; gap=14
        agg_col={"M1":C["blue"],"M2":C["teal"],"M3":C["amber"]}
        for (lab,hl),tag in zip(items,tags):
            yc=y-h
            face=C["rose"] if hl else C["card"]
            edge=C["rose"] if hl else C["edge"]
            nm.card(ax,cx-330,yc,660,h,edge=edge,fill=(C["bg"] if not hl else None),lw=2.6 if hl else 2.0,radius=14)
            ax.text(cx-300,yc+h/2,lab,ha="left",va="center",fontsize=18.5,
                    color=(C["rose"] if hl else C["text"]),fontweight="bold" if hl else "normal")
            # tag agrégat à droite
            ax.text(cx+300,yc+h/2,tag,ha="right",va="center",fontsize=17,color=agg_col[tag],fontweight="bold")
            y=yc-gap
    col(W*0.27,t[2],t[4],t[6])
    col(W*0.73,t[3],t[5],t[7])
    nm.footer(fig,t[8]);
    return fig


fig = build_figure(LANG)